In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, initcap, from_json, concat_ws, lit
from pyspark.sql.types import StringType, StructType, StructField

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"
# Read drivers data from bronze layer
bronze_drivers_df = spark.table(bronze_table)


In [0]:
# bronze_drivers_df.display()

In [0]:
# Apply transformations
silver_drivers_df = (
    bronze_drivers_df
    # Remove records with NULL constructor_id
    .filter(col("driverId").isNotNull())
    # Remove duplicates
    .dropDuplicates(["driverId"])
    # Drop URL column
    .drop("url")
    # Rename columns to snake_case and meaningful names
    .withColumnRenamed("driverId", "driver_id")
    .withColumnRenamed("dateOfBirth", "date_of_birth")
    .withColumn(
        "driver_name", initcap(concat_ws(" ", col("name.familyName"), col("name.givenName")))
    )
    .drop("name")
    # Transform nationality to Title Case))
    # Transform nationality to Title Case
    .withColumn("nationality", initcap(col("nationality")))
    .select(
        "driver_id",
        "date_of_birth",
        "driver_name",
        "nationality",
        "ingestion_timestamp",
        "source_file_path",
    )
)

# print(f"Total records after transformations: {silver_constructors_df.count()}")
# display(silver_constructors_df)

In [0]:
# display(silver_drivers_df)